In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [2]:
# Load all tickers
with open('../data/processed/ticker_list.txt', 'r') as f:
    tickers = [line.strip() for line in f.readlines()]

# Load all data
dfs = []
for ticker in tickers:
    try:
        df = pd.read_csv(f'../data/processed/{ticker}_features.csv')
        dfs.append(df)
    except:
        pass

data = pd.concat(dfs, ignore_index=True)
data['Date'] = pd.to_datetime(data['Date'])

# 2 year filter
data = data[data['Date'] >= pd.Timestamp('2024-01-01')]

# Feature columns
drop_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume',
             'Trades', 'Turnover', 'Ticker', 'Label', 'Future_Return', 'Sector']
feature_cols = [c for c in data.columns if c not in drop_cols]

print(f"Total rows: {len(data)}")
print(f"Features: {len(feature_cols)}")
print(f"Sectors: {data['Sector'].nunique()}")
print(data['Sector'].value_counts())

Total rows: 139651
Features: 23
Sectors: 12
Sector
Hydro Power                     52598
Microfinance                    27223
Commercial Banks                10792
Finance                          8518
Development Banks                8452
Life Insurance                   7397
Non Life Insurance               7088
Manufacturing And Processing     4913
Others                           4008
Investment                       3975
Hotels And Tourism               3553
Tradings                         1134
Name: count, dtype: int64


In [3]:
# Define target sectors (enough data)
target_sectors = [
    'Hydro Power',
    'Microfinance',
    'Commercial Banks',
    'Development Banks',
    'Finance',
    'Life Insurance',
    'Manufacturing And Processing',
    'Non Life Insurance',
]

results = {}

for sector in target_sectors:
    print(f"\n{'='*45}")
    print(f"Training: {sector}")

    # Filter sector data
    sector_data = data[data['Sector'] == sector].copy()
    sector_data = sector_data.dropna(subset=feature_cols + ['Label'])

    n_stocks = sector_data['Ticker'].nunique()
    n_rows = len(sector_data)
    print(f"Stocks: {n_stocks} | Rows: {n_rows}")

    X = sector_data[feature_cols].values
    y = sector_data['Label'].values

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train XGBoost
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
    xgb.fit(X_train, y_train)

    # Evaluate
    preds = xgb.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Accuracy: {acc*100:.1f}%")
    print(classification_report(y_test, preds, target_names=['Sell', 'Hold', 'Buy']))

    results[sector] = {
        'accuracy': round(acc, 4),
        'stocks': n_stocks,
        'rows': n_rows,
        'model': xgb
    }

# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*55)
print("XGBOOST SECTOR SUMMARY")
print("="*55)
for sector, res in results.items():
    print(f"{sector:35} | Acc: {res['accuracy']*100:.1f}% | Stocks: {res['stocks']}")


Training: Hydro Power
Stocks: 97 | Rows: 52088
Accuracy: 47.4%
              precision    recall  f1-score   support

        Sell       0.44      0.39      0.41      3315
        Hold       0.50      0.72      0.59      4269
         Buy       0.43      0.21      0.29      2834

    accuracy                           0.47     10418
   macro avg       0.46      0.44      0.43     10418
weighted avg       0.46      0.47      0.45     10418


Training: Microfinance
Stocks: 50 | Rows: 26978
Accuracy: 50.3%
              precision    recall  f1-score   support

        Sell       0.44      0.27      0.34      1541
        Hold       0.53      0.81      0.64      2497
         Buy       0.42      0.19      0.27      1358

    accuracy                           0.50      5396
   macro avg       0.47      0.43      0.41      5396
weighted avg       0.48      0.50      0.46      5396


Training: Commercial Banks
Stocks: 19 | Rows: 10792
Accuracy: 62.7%
              precision    recall  f1-sc